<a href="https://colab.research.google.com/github/ekt4r/playground-series-s6e3/blob/main/xgboost_tuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train = pd.read_csv('/content/drive/MyDrive/playground-series-s6e3/train.csv').drop('id', axis=1)
test = pd.read_csv('/content/drive/MyDrive/playground-series-s6e3/test.csv').drop('id', axis=1)
sample_submission = pd.read_csv('/content/drive/MyDrive/playground-series-s6e3/sample_submission.csv')

In [4]:
for df in [train, test]:

    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

    # df['MCSquared'] = df['MonthlyCharges'] ** 2
    # df['TCSquared'] = df['TotalCharges'] ** 2

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    # df['MCSqrt'] = np.sqrt(df['MonthlyCharges'])
    # df['TCSqrt'] = np.sqrt(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    # Loyalty
    df["TenureLog"] = np.log1p(df["tenure"])
    # df["TenureSquared"] = df["tenure"]**2
    # df['TenureSqrt'] = np.sqrt(df['tenure'])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    # Spending
    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]

    # Contract
    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    # Payment
    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    # Service Engagement
    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    # Internet
    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    # Early Risk
    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    # Value
    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

train['MCGT75%'] = train['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)
test['MCGT75%'] = test['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)

train['TCGT75%'] = train['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)
test['TCGT75%'] = test['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)

/tmp/ipykernel_195/4109498660.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)
/tmp/ipykernel_195/4109498660.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(v

In [5]:
X, y = train.drop('Churn', axis=1), train['Churn']
y = y.map({'Yes': 1, 'No': 0})

In [6]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['Contract', 'PaymentMethod']

ord_mapping = {
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)

In [7]:
!nvidia-smi

Sat Mar  7 02:39:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [18]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded[train_idx], X_train_encoded[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = XGBClassifier(
        n_estimators=20000,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        early_stopping_rounds=200,
        eval_metric="auc",
        device='cuda'
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=200
    )
    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

Fold 1
[0]	validation_0-auc:0.90796
[200]	validation_0-auc:0.91497
[400]	validation_0-auc:0.91579
[600]	validation_0-auc:0.91590
[800]	validation_0-auc:0.91582
[830]	validation_0-auc:0.91580


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [02:46:07] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Fold 2
[0]	validation_0-auc:0.90878
[200]	validation_0-auc:0.91604
[400]	validation_0-auc:0.91686
[600]	validation_0-auc:0.91696
[728]	validation_0-auc:0.91693
Fold 3
[0]	validation_0-auc:0.90853
[200]	validation_0-auc:0.91552
[400]	validation_0-auc:0.91635
[600]	validation_0-auc:0.91644
[743]	validation_0-auc:0.91642
Fold 4
[0]	validation_0-auc:0.91006
[200]	validation_0-auc:0.91647
[400]	validation_0-auc:0.91737
[600]	validation_0-auc:0.91749
[761]	validation_0-auc:0.91744
Fold 5
[0]	validation_0-auc:0.90645
[200]	validation_0-auc:0.91369
[400]	validation_0-auc:0.91456
[600]	validation_0-auc:0.91466
[800]	validation_0-auc:0.91455
[811]	validation_0-auc:0.91454


In [19]:
from sklearn.metrics import roc_auc_score

cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9162935714911312


In [20]:
submission = sample_submission.copy()
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)

In [21]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [22]:
!kaggle competitions submit -c playground-series-s6e3 -f submission.csv  -m "XGBoost CV5 GPU"

100% 6.61M/6.61M [00:00<00:00, 8.87MB/s]
Successfully submitted to Predict Customer Churn

In [23]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 9.4 MB/s eta 0:00:00


In [24]:
import optuna
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def objective(trial):

    params = {
        "n_estimators": 5000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
        "random_state": 42,
        "eval_metric": "auc",
        "device": "cuda",
        "early_stopping_rounds": 200
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof = np.zeros(len(X_train_encoded))

    for train_idx, valid_idx in skf.split(X_train_encoded, y):

        X_train, X_valid = X_train_encoded[train_idx], X_train_encoded[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]

        model = XGBClassifier(**params)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            verbose=False
        )

        preds = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = preds

    score = roc_auc_score(y, oof)

    return score

In [25]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(objective, n_trials=50)

[I 2026-03-07 02:51:16,899] A new study created in memory with name: no-name-704dadb8-2a97-4e27-a10a-c651434dc9c1
[I 2026-03-07 02:52:50,929] Trial 0 finished with value: 0.9158493441999767 and parameters: {'learning_rate': 0.011933192475821326, 'max_depth': 6, 'min_child_weight': 4, 'subsample': 0.9403880083609867, 'colsample_bytree': 0.9221900196442638, 'gamma': 3.369917965405894, 'reg_alpha': 1.431499319486641, 'reg_lambda': 6.22714987810948}. Best is trial 0 with value: 0.9158493441999767.
[I 2026-03-07 02:53:42,482] Trial 1 finished with value: 0.9168391377304222 and parameters: {'learning_rate': 0.059435809066431415, 'max_depth': 4, 'min_child_weight': 9, 'subsample': 0.8341346857726095, 'colsample_bytree': 0.520159184338049, 'gamma': 0.7102328700195942, 'reg_alpha': 6.139222908478064, 'reg_lambda': 5.5798577643626714}. Best is trial 1 with value: 0.9168391377304222.
[I 2026-03-07 02:54:03,268] Trial 2 finished with value: 0.9159346713036068 and parameters: {'learning_rate': 0.14

In [26]:
import json

with open('xgboost-params.json', 'w') as json_file:
    json.dump(study.best_params, json_file, indent=4)

In [28]:
print(study.best_trial)
print(study.best_params)

FrozenTrial(number=17, state=<TrialState.COMPLETE: 1>, values=[0.9168893501284422], datetime_start=datetime.datetime(2026, 3, 7, 3, 9, 35, 116570), datetime_complete=datetime.datetime(2026, 3, 7, 3, 11, 22, 366127), params={'learning_rate': 0.022200439192185714, 'max_depth': 4, 'min_child_weight': 7, 'subsample': 0.8852275843032212, 'colsample_bytree': 0.5048287064762882, 'gamma': 0.6532478645748768, 'reg_alpha': 5.190282638441493, 'reg_lambda': 2.7585060147705223}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.2, log=True, low=0.01, step=None), 'max_depth': IntDistribution(high=10, log=False, low=3, step=1), 'min_child_weight': IntDistribution(high=10, log=False, low=1, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.5, step=None), 'gamma': FloatDistribution(high=5.0, log=False, low=0.0, step=None), 'reg_alpha': FloatDist

In [29]:
best_params = study.best_params

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded[train_idx], X_train_encoded[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = XGBClassifier(
        **best_params,
        n_estimators=5000,
        eval_metric="auc",
        device="cuda",
        early_stopping_rounds=200
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=200
    )
    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

Fold 1
[0]	validation_0-auc:0.89065
[200]	validation_0-auc:0.91219
[400]	validation_0-auc:0.91403
[600]	validation_0-auc:0.91488
[800]	validation_0-auc:0.91535
[1000]	validation_0-auc:0.91566
[1200]	validation_0-auc:0.91589
[1400]	validation_0-auc:0.91605
[1600]	validation_0-auc:0.91616
[1800]	validation_0-auc:0.91625
[2000]	validation_0-auc:0.91632
[2200]	validation_0-auc:0.91637
[2400]	validation_0-auc:0.91642
[2600]	validation_0-auc:0.91645
[2800]	validation_0-auc:0.91648
[3000]	validation_0-auc:0.91650
[3200]	validation_0-auc:0.91652
[3400]	validation_0-auc:0.91653
[3600]	validation_0-auc:0.91653
[3800]	validation_0-auc:0.91654
[4000]	validation_0-auc:0.91655
[4200]	validation_0-auc:0.91655
[4400]	validation_0-auc:0.91656
[4600]	validation_0-auc:0.91657
[4800]	validation_0-auc:0.91658
[4999]	validation_0-auc:0.91658
Fold 2
[0]	validation_0-auc:0.89204
[200]	validation_0-auc:0.91327
[400]	validation_0-auc:0.91501
[600]	validation_0-auc:0.91583
[800]	validation_0-auc:0.91628
[1000]	v

In [30]:
cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9168885522145158


In [31]:
submission = sample_submission.copy()
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)

In [32]:
!kaggle competitions submit -c playground-series-s6e3 -f submission.csv  -m "XGBoost CV5 Finetuned"

100% 6.60M/6.60M [00:00<00:00, 9.20MB/s]
Successfully submitted to Predict Customer Churn